# Lahore Night Lights (VIIRS Monthly)
## Export: 500 m radiance GeoTIFF and 250 m point GeoJSON


### 0. Initialize Earth Engine


In [1]:
import calendar
import urllib.request

import ee, geemap, geopandas as gpd

try:
    ee.Initialize()
except Exception:
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize()


/Users/ahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### 1. Parameters


In [2]:
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"
YEAR = 2026
MONTH = 2
last_day = calendar.monthrange(YEAR, MONTH)[1]
START = f"{YEAR:04d}-{MONTH:02d}-01"
END = f"{YEAR:04d}-{MONTH:02d}-{last_day:02d}"

GEOTIFF_SCALE = 500
POINT_SCALE = 250
SMOOTH_RADIUS_M = 0  # e.g., 500 to soften
OUT_PREFIX = f"VIIRS_NL_Lahore_{calendar.month_abbr[MONTH]}{YEAR}"

print(f"Using date range: {START} to {END}")
print(f"Output prefix: {OUT_PREFIX}")

Using date range: 2026-02-01 to 2026-02-28
Output prefix: VIIRS_NL_Lahore_Feb2026


### 2. Load Lahore Boundary and Build Monthly VIIRS Composite


In [3]:
gdf = gpd.read_file(UC_SHP)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)
ucs_fc = geemap.gdf_to_ee(gdf)
region = ucs_fc.geometry()

DATASETS = [
    "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG",
    "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG",
    "NOAA/VIIRS/DNB/MONTHLY_V2/VCMSLCFG",
    "NOAA/VIIRS/DNB/MONTHLY_V21/VCMSLCFG",
]

def get_viirs_monthly():
    for ds in DATASETS:
        try:
            ee.ImageCollection(ds).limit(1).size().getInfo()
            return ds
        except Exception:
            continue
    raise RuntimeError("No VIIRS monthly collection found.")

VIIRS = get_viirs_monthly()
print(f"[OK] Using: {VIIRS}")

def prep_viirs(img):
    rad = img.select("avg_rad")
    rad = rad.updateMask(rad.gte(0))
    band_names = img.bandNames()
    has_cvg = band_names.contains("cf_cvg")
    cvg_mask = ee.Image(ee.Algorithms.If(
        has_cvg,
        img.select("cf_cvg").gt(0),
        ee.Image(1),
    ))
    return rad.updateMask(cvg_mask).rename("rad")

col = (
    ee.ImageCollection(VIIRS)
    .filterBounds(region)
    .filterDate(START, END)
    .map(prep_viirs)
)

nl_med = col.median().rename("rad")

if SMOOTH_RADIUS_M > 0:
    kernel = ee.Kernel.circle(radius=SMOOTH_RADIUS_M, units="meters", normalize=True)
    nl_med = nl_med.focal_mean(kernel=kernel, iterations=1)


[OK] Using: NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG


### 3. Export GeoTIFF and Point GeoJSON


In [4]:
def export_image_local(img, filename, region, scale=500, crs="EPSG:4326"):
    try:
        geemap.ee_export_image(img.clip(region), filename, scale=scale, region=region, file_per_band=False)
        print(f"[OK] GeoTIFF -> {filename}")
        return
    except TypeError:
        pass
    except Exception as e:
        print("[INFO] ee_export_image failed:", e)
    try:
        geemap.download_ee_image(img.clip(region), filename=filename, scale=scale, region=region, crs=crs)
        print(f"[OK] GeoTIFF (download_ee_image) -> {filename}")
        return
    except Exception as e:
        print("[INFO] download_ee_image failed:", e)
    url = img.clip(region).getDownloadURL({
        "scale": scale,
        "crs": crs,
        "region": region,
        "filePerBand": False,
    })
    urllib.request.urlretrieve(url, filename)
    print(f"[OK] GeoTIFF (getDownloadURL) -> {filename}")

tif_path = f"{OUT_PREFIX}_rad_{GEOTIFF_SCALE}m.tif"
export_image_local(nl_med, tif_path, region, scale=GEOTIFF_SCALE)

pts_fc = ee.Image.pixelLonLat().addBands(nl_med.rename("val")).sample(
    region=region,
    scale=POINT_SCALE,
    geometries=True,
    seed=1,
)

geojson_pts = f"{OUT_PREFIX}_points_{POINT_SCALE}m.geojson"
geemap.ee_export_vector(pts_fc, filename=geojson_pts)
print(f"[OK] GeoJSON points -> {geojson_pts}")

Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/NL/VIIRS_NL_Lahore_Feb2026_rad_500m.tif
[OK] GeoTIFF -> VIIRS_NL_Lahore_Feb2026_rad_500m.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/ahmed/Desktop/Senior Spring 26/SPROJ - Dr Tahir/SPROJ/notebooks/NL/VIIRS_NL_Lahore_Feb2026_points_250m.geojson
[OK] GeoJSON points -> VIIRS_NL_Lahore_Feb2026_points_250m.geojson
